In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# 1. Load the 3-class dataset directly from URL
dataset_url = "https://raw.githubusercontent.com/satyamk28/Twitter-Sentiment-Analysis/master/twitter_training.csv"
df = pd.read_csv(dataset_url, names=['id', 'entity', 'sentiment', 'text'])

# 2. Drop missing data and filter target classes
df = df.dropna(subset=['text', 'sentiment'])
df = df[['text', 'sentiment']]
df['sentiment'] = df['sentiment'].str.lower()
df = df[df['sentiment'].isin(['positive', 'negative', 'neutral'])]

# Map labels to numbers
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_mapping)

# 3. Text cleaning function
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

# Downsample to 15,000 rows
df_sample = df.sample(n=15000, random_state=42).reset_index(drop=True)
df_sample['cleaned_text'] = df_sample['text'].apply(clean_text)

# 4. Tokenization & Padding
max_words = 5000
max_len = 35

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df_sample['cleaned_text'])

sequences = tokenizer.texts_to_sequences(df_sample['cleaned_text'])
X = pad_sequences(sequences, maxlen=max_len, padding='post')
y = to_categorical(df_sample['label'], num_classes=3)

# 5. Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data pipeline set up! Prepared training shapes:", X_train.shape)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Data pipeline set up! Prepared training shapes: (12000, 35)


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

embedding_dim = 100

model_cnn = Sequential([
    # 1. Embedding Layer
    Embedding(input_dim=max_words, output_dim=embedding_dim),

    # 2. Convolutional Layer (extracts feature patterns)
    Conv1D(filters=128, kernel_size=5, activation='relu'),

    # 3. Pooling Layer (dimensionality reduction)
    GlobalMaxPooling1D(),

    # Regularization layer to prevent overfitting
    Dropout(0.3),

    # 4. Fully Connected Output Layer
    Dense(units=3, activation='softmax')
])

model_cnn.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [3]:
from sklearn.metrics import classification_report, accuracy_score

# Train the model (CNNs train noticeably faster than LSTMs!)
print("Training the CNN Text Classifier...")
history = model_cnn.fit(X_train, y_train, epochs=4, batch_size=32, validation_split=0.1, verbose=1)

# Evaluate model performance
print("\nEvaluating model performance...")
y_pred_probs = model_cnn.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Print metrics
print(f"\nOverall CNN Model Accuracy: {accuracy_score(y_true, y_pred):.4f}\n")
print("Detailed Performance Report:")
print(classification_report(y_true, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

Training the CNN Text Classifier...
Epoch 1/4
338/338 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.5612 - loss: 0.9162 - val_accuracy: 0.6883 - val_loss: 0.7276
Epoch 2/4
338/338 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.7598 - loss: 0.5965 - val_accuracy: 0.7267 - val_loss: 0.6777
Epoch 3/4
338/338 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.8515 - loss: 0.3897 - val_accuracy: 0.7400 - val_loss: 0.6795
Epoch 4/4
338/338 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9044 - loss: 0.2496 - val_accuracy: 0.7517 - val_loss: 0.7491

Evaluating model performance...
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

Overall CNN Model Accuracy: 0.7270

Detailed Performance Report:
              precision    recall  f1-score   support

    Negative       0.77      0.77      0.77      1082
     Neutral       0.68      0.67      0.67       884
    Positive       0.72      0.73      0.73      1034

    accuracy                           0.73      3000
   macro avg       0.72      0.72      0.7